# Fightin' Words: Fox News vs Just Women's Sports

Which words does each outlet use *more than the other*, beyond chance?
Method: weighted log-odds with an informative Dirichlet prior
(Monroe, Colaresi & Quinn 2008), via Jack Hessel's implementation
(`fighting_words.py`, vendored in this folder).

Everything runs from the CSVs in `../data/` — no pipeline needed.

In [ ]:
import re
import pandas as pd
from fighting_words import bayes_compare_language

fox = pd.read_csv("../data/fox_wnba.csv")
jws = pd.read_csv("../data/just_womens_sports.csv")
print(len(fox), "Fox articles ·", len(jws), "JWS articles")

In [ ]:
# keep it simple: lowercase words, drop a few obvious non-words
def clean(text):
    words = re.findall(r"[a-z]{3,}", str(text).lower())
    return " ".join(words)

fox_docs = (fox["title"].fillna("") + " " + fox["body"].fillna("")).map(clean)
jws_docs = (jws["headline"].fillna("") + " " + jws["body"].fillna("")).map(clean)

In [ ]:
# z > 0 means "more Fox than JWS"; z < 0 the reverse.
# |z| > 1.96 is the usual significance line.
scored = bayes_compare_language(list(fox_docs), list(jws_docs), ngram=1, prior=0.05)
words = pd.DataFrame(scored, columns=["word", "z"]).sort_values("z")

print("MOST FOX ------------------------")
print(words.tail(25).iloc[::-1].to_string(index=False))
print()
print("MOST JWS ------------------------")
print(words.head(25).to_string(index=False))

The Fox side of this list is the piece's core finding: discourse words
(*media, people, think, wrote, foul*) instead of basketball words
(*coach, roster, points*). Try swapping in `espn.csv` or `fox_nba.csv`
to reproduce the other comparisons.